In [ ]:
from pathlib import Path
from scipy.ndimage import zoom, find_objects
import torchio as tio
import os
import re
import nibabel as nib
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
import json
import torch.multiprocessing as mp
from fvcore.nn import FlopCountAnalysis, parameter_count

# ==================== 全局随机种子设置（保证实验可复现） ====================
SEED = 42  # 固定种子值，可根据需要修改
# 设置Python内置随机种子
import random
random.seed(SEED)
# 设置numpy随机种子
np.random.seed(SEED)
# 设置PyTorch CPU随机种子
torch.manual_seed(SEED)
# 设置PyTorch GPU随机种子（多卡也生效）
torch.cuda.manual_seed_all(SEED)
# 确保CUDA卷积算法的确定性（牺牲少量速度换可复现性）
torch.backends.cudnn.deterministic = True
# 关闭CUDA卷积算法的自动优化（避免随机选择算法）
torch.backends.cudnn.benchmark = False

# ==================== 全局配置 ====================
CLASS_MAP = {'NOR':0, 'DCM':1, 'HCM':2, 'MINF':3, 'RV':4}
TARGET_SHAPE = (200, 200, 80)
TARGET_SPACING = 1.25  # mm
AUG_FACTOR = 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==================== 用fvcore实现复杂度分析函数 ====================
def calculate_model_complexity(model, device):

    # 创建 dummy 输入（双分支单通道，和真实输入维度一致）
    dummy_ed = torch.randn(1, 1, *TARGET_SHAPE).to(device)  # [B=1, 1通道, 200, 200, 80]
    dummy_es = torch.randn(1, 1, *TARGET_SHAPE).to(device)  # [B=1, 1通道, 200, 200, 80]
    dummy_ef = torch.tensor([0.5], dtype=torch.float32).to(device)  # 随机EF值
    
    # 计算参数量（fvcore的parameter_count返回字典，""对应总参数量）
    params_dict = parameter_count(model)
    params = params_dict[""]  # 总参数量
    params_m = params / 1e6  # 转换为 MParams
    
    # 计算FLOPs（fvcore的FlopCountAnalysis，支持更复杂的操作）
    model.eval()  # 切换到评估模式避免Dropout等影响
    with torch.no_grad():
        flops_analyzer = FlopCountAnalysis(model, (dummy_ed, dummy_es, dummy_ef))
        flops_analyzer = flops_analyzer.unsupported_ops_warnings(False)  # 关闭不支持操作的警告
        flops = flops_analyzer.total()
        flops_g = flops / 1e9  
    
    print("\n" + "="*60)
    print(f"📊 模型复杂度分析（fvcore）")
    print(f"   参数量 (Params): {params_m:.2f} M")
    print(f"   计算量 (FLOPs):  {flops_g:.2f} G")
    print("="*60 + "\n")
    
    return params, flops

class RotaryEmbedding3D(nn.Module):

    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        assert dim % 6 == 0, "特征维度必须是6的倍数（每个轴分配2*k维复数对）"
        self.d_per_axis = dim // 3  # 每个轴分配的维度

    def get_sin_cos(self, n, d_axis, device):
        """生成单个轴的正弦/余弦旋转矩阵"""
        pos = torch.arange(n, device=device, dtype=torch.float32)
        inv_freq = 1.0 / (10000 ** (torch.arange(0, d_axis, 2, device=device).float() / d_axis))
        sinusoid_inp = torch.einsum("i,j->ij", pos, inv_freq)
        return sinusoid_inp.sin(), sinusoid_inp.cos()

    def apply_rotary(self, t, sin, cos):
        """对特征张量应用旋转变换（复数对旋转）"""
        t_real, t_imag = t[..., 0::2], t[..., 1::2]
        res_real = t_real * cos - t_imag * sin
        res_imag = t_real * sin + t_imag * cos
        res = torch.stack([res_real, res_imag], dim=-1).flatten(-2)
        return res

    def forward(self, x, spatial_shape):
        # x: [B, p, i, m] -> p=D'*H'*W'（PrimaryCaps3D卷积后的展平维度）
        # spatial_shape: (D', H', W') -> PrimaryCaps3D输出的真实空间维度
        B, p, i, m = x.shape
        D, H, W = spatial_shape
        device = x.device
        
        # 校验：展平维度需等于PrimaryCaps3D输出的空间维度乘积
        assert p == D*H*W, f"展平维度p={p}与PrimaryCaps3D输出的空间维度D*H*W={D*H*W}不匹配"
        
        # 将展平的空间维度还原为3D
        x = x.view(B, D, H, W, i, m)

        # 生成三个轴的旋转矩阵
        sd, cd = self.get_sin_cos(D, self.d_per_axis, device)
        sh, ch = self.get_sin_cos(H, self.d_per_axis, device)
        sw, cw = self.get_sin_cos(W, self.d_per_axis, device)

        # 对三个轴的特征分别应用旋转（维度广播匹配）
        x_d = self.apply_rotary(x[..., :self.d_per_axis], sd.view(1, D, 1, 1, 1, -1), cd.view(1, D, 1, 1, 1, -1))
        x_h = self.apply_rotary(x[..., self.d_per_axis:2*self.d_per_axis], sh.view(1, 1, H, 1, 1, -1), ch.view(1, 1, H, 1, 1, -1))
        x_w = self.apply_rotary(x[..., 2*self.d_per_axis:], sw.view(1, 1, 1, W, 1, -1), cw.view(1, 1, 1, W, 1, -1))

        # 合并旋转后的特征并展平空间维度
        out = torch.cat([x_d, x_h, x_w], dim=-1)
        return out.view(B, p, i, m)

class InteractiveRotaryCapsule3D(nn.Module):

    def __init__(self, in_caps, out_caps, in_dim, out_dim, num_routing=3):
        super().__init__()
        self.num_routing = num_routing
        self.out_caps = out_caps
        
        # 路由变换矩阵（正交初始化保证稳定性）
        self.W = nn.Parameter(torch.Tensor(in_caps, out_caps, in_dim, out_dim))
        nn.init.orthogonal_(self.W)
        
        # 局部交互算子：分组卷积+BN+Sigmoid，增强空间一致性
        self.spatial_interact = nn.Sequential(
            nn.Conv3d(out_caps, out_caps, kernel_size=3, padding=1, groups=out_caps),
            nn.BatchNorm3d(out_caps),
            nn.Sigmoid()
        )

    def squash(self, input_tensor):
        """胶囊网络非线性压缩函数"""
        norm = torch.norm(input_tensor, dim=-1, keepdim=True)
        return (norm**2 / (1 + norm**2)) * (input_tensor / (norm + 1e-8))

    def forward(self, u, spatial_shape):
        # u: [B, p, i, m] -> 经3D-RCPE注入空间信息的胶囊向量
        B, p, i, m = u.shape
        D, H, W = spatial_shape
        device = u.device
        
        # 1. 计算预测向量（解耦计算：空间维度独立）
        u_hat = torch.einsum('bpim,ijmn->bpijn', u, self.W)  # [B, p, i, j, n]
        
        # 2. 初始化路由logits
        b = torch.zeros(B, p, i, self.out_caps, device=device)
        
        for r in range(self.num_routing):
            # 路由耦合系数（softmax归一化）
            c = F.softmax(b, dim=-1)  # [B, p, i, j]
            
            # 加权求和（解耦计算：空间维度独立求和）
            s = torch.einsum('bpij,bpijn->bjn', c, u_hat)  # [B, j, n]
            v = self.squash(s)  # [B, j, n]

            if r < self.num_routing - 1:
                # 协议值计算：预测向量与胶囊输出的内积
                agreement = torch.einsum('bpijn,bjn->bpij', u_hat, v)  # [B, p, i, j]
                
                # --- 核心改进：局部交互校验 ---
                # 步骤1：协议值投影回3D空间 [B, p, i, j] -> [B*i, j, D, H, W]
                agree_spatial = agreement.permute(0, 2, 3, 1).reshape(B*i, self.out_caps, D, H, W)
                # 步骤2：3D卷积做局部邻域平滑
                interact_factor = self.spatial_interact(agree_spatial)
                # 步骤3：投影回路由维度 [B*i, j, D, H, W] -> [B, p, i, j]
                interact_factor = interact_factor.reshape(B, i, self.out_caps, p).permute(0, 3, 1, 2)
                
                # 路由更新：协议值 × 空间交互因子（融合空间一致性）
                b = b + agreement * interact_factor
                
        return v

# ==================== PrimaryCaps3D ====================
class PrimaryCaps3D(nn.Module):
    def __init__(self, in_channels=252, caps_dim=6, kernel_size=3, stride=2): 
        super().__init__()
        self.caps_dim = caps_dim
        self.out_caps = in_channels // caps_dim  # 主胶囊数量
        self.conv = nn.Conv3d(in_channels, self.out_caps * caps_dim, 
                            kernel_size, stride=stride, padding=1)
      
    def forward(self, x):
        batch_size = x.size(0)
        out = self.conv(x)  # [B, out_caps*caps_dim, D', H', W'] -> PrimaryCaps3D卷积后的空间维度
        # 记录PrimaryCaps3D输出的空间维度（关键修复点）
        self.spatial_shape = (out.shape[2], out.shape[3], out.shape[4])
        
        # 重塑维度：分离胶囊维度和特征维度
        out = out.view(batch_size, self.out_caps, self.caps_dim,
                      out.size(-3), out.size(-2), out.size(-1))  # [B, out_caps, caps_dim, D', H', W']
      
        # 调整维度顺序并展平空间维度
        out = out.permute(0, 3, 4, 5, 1, 2).contiguous()  # [B, D', H', W', out_caps, caps_dim]
        spatial_flatten = out.size(1)*out.size(2)*out.size(3)  # p = D'*H'*W'
        # 修复：返回胶囊向量 + 空间维度
        return out.view(batch_size, spatial_flatten, self.out_caps, self.caps_dim), self.spatial_shape

class EFGatedAttention(nn.Module):

    def __init__(self, caps_dim=6):
        super().__init__()
        self.gate_net = nn.Sequential(
            nn.Linear(1, 16),
            nn.GELU(),
            nn.Linear(16, caps_dim),
            nn.Sigmoid()  # 输出[0,1]区间
        )
        self.layer_norm = nn.LayerNorm(caps_dim)

    def forward(self, capsules, ef):
        batch_size = capsules.size(0)
        # EF特征门控：广播到所有胶囊
        gate = self.gate_net(ef.unsqueeze(1))  # [B, caps_dim]
        gate = gate.view(batch_size, 1, 1, -1)  # [B, 1, 1, caps_dim]
        # 残差连接+层归一化增强稳定性
        gated_caps = capsules * gate
        return self.layer_norm(gated_caps + capsules)

# ==================== 整合双分支共享编码器 + RCPE随机深度 ====================
class Caps3DNet(nn.Module):

    def __init__(self, num_classes=5, caps_dim=6, num_routing=3,
                 RCPE_drop_prob=0.1): # 添加RCPE随机深度参数
        super().__init__()
        self.caps_dim = caps_dim
        self.num_routing = num_routing
        
        # RCPE随机深度配置
        self.RCPE_drop_prob = RCPE_drop_prob
        self.last_RCPE_keep_ratio = None  # 仅用于日志观察
        
        # 1. 共享特征提取编码器（单通道输入，ED/ES共享权重）
        self.conv3d = nn.Sequential(
            nn.Conv3d(1, 252, kernel_size=9, stride=3),
            nn.BatchNorm3d(252),
            nn.ReLU(),
            nn.Conv3d(252, 252, kernel_size=3, stride=2, padding=1),
            nn.Dropout3d(0.2)
        )
        
        # 2. 通道适配层：融合后756通道 → 252通道，适配后续胶囊层
        self.channel_adapter = nn.Sequential(
            nn.Conv3d(756, 252, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm3d(252),
            nn.ReLU()
        )
        
        # 3. 初级胶囊层：252D->胶囊向量 [B, p, 42, 6]（252//6=42个胶囊，完全整除）
        self.primary_caps = PrimaryCaps3D(in_channels=252, caps_dim=caps_dim)
        
        # 4. 3D-RCPE：注入空间拓扑信息
        self.RCPE = RotaryEmbedding3D(dim=caps_dim)
        
        # 5. EF门控注意力：基于射血分数的特征门控
        self.ef_gate = EFGatedAttention(caps_dim=caps_dim)
        
        # 6. 解耦-交互旋转路由层：42->5个分类胶囊，6->12维特征
        self.digit_caps = InteractiveRotaryCapsule3D(
            in_caps=42,  # 显式写42，替代256//caps_dim
            out_caps=num_classes,   # 5个输出分类胶囊
            in_dim=caps_dim,        # 输入特征维度6
            out_dim=12,             # 输出特征维度12
            num_routing=num_routing
        )
        
        # 7. 分类器：5*12->5维分类输出
        self.classifier = nn.Linear(num_classes * 12, num_classes)

    # 添加获取RCPE保留率的方法
    def get_RCPE_keep_ratio(self):
        if self.last_RCPE_keep_ratio is None:
            return None
        return self.last_RCPE_keep_ratio

    def forward(self, ed_x, es_x, ef):
        batch_size = ed_x.size(0)
        
        # 1. 共享编码器分别提取ED、ES单时相特征
        ed_feat = self.conv3d(ed_x)
        es_feat = self.conv3d(es_x)
        
        # 2. ED + ES without Diff 消融设置：
        #    保留ED/ES双时相特征，但将ES-ED时相差分分支用零张量屏蔽。
        #    这样可以保持channel_adapter输入通道数(756)和后续胶囊网络结构不变。
        feat_diff = torch.zeros_like(ed_feat)
        fused_feat = torch.cat([ed_feat, es_feat, feat_diff], dim=1)
        
        # 3. 通道适配，对齐后续胶囊层输入维度
        x = self.channel_adapter(fused_feat)
        
        # 4. 初级胶囊化：返回胶囊向量 + 真实空间维度
        u, spatial_shape = self.primary_caps(x)
        
        # RCPE残差分支随机深度
        u_RCPE = self.RCPE(u, spatial_shape)
        RCPE_residual = u_RCPE - u

        if self.training and self.RCPE_drop_prob > 0:
            keep_prob = 1.0 - self.RCPE_drop_prob

            # 每个样本一个mask，shape [B,1,1,1]
            mask = torch.bernoulli(
                torch.full((u.size(0), 1, 1, 1), keep_prob, device=u.device)
            ) / keep_prob

            self.last_RCPE_keep_ratio = (mask > 0).float().mean().detach().item()

            u = u + mask * RCPE_residual
        else:
            # 验证/测试阶段，或者drop_prob=0时，完整使用RCPE
            self.last_RCPE_keep_ratio = 1.0
            u = u + RCPE_residual  # 等价于 u_RCPE
        
        # 6. EF门控注意力
        u = self.ef_gate(u, ef)
        
        # 7. 解耦-交互旋转路由
        caps_output = self.digit_caps(u, spatial_shape)  # [B, j, n]
        
        # 8. 分类输出
        return self.classifier(caps_output.view(batch_size, -1))

# ==================== 数据集模块 ====================
class PairwiseAugmentor(tio.Transform):
    """ED/ES时相同步增强占位符"""
    def __init__(self):
        super().__init__()
    def apply_transform(self, subject):
        return subject

def medical_intensity_augmentation(tensor):
    """医学图像强度增强函数"""
    # 伽马校正
    gamma = torch.FloatTensor(1).uniform_(0.8, 1.2)
    tensor = tensor.sign() * (tensor.abs() ** gamma.item())
  
    # 局部对比度扰动
    if torch.rand(1) > 0.6:
        block = tio.RandomNoise(std=(0, 0.1))(tensor.clone())
        mask = tio.RandomBlur(std=(2, 4))(torch.rand_like(tensor) > 0.85)
        tensor = tensor * (1 - mask) + block * mask
  
    # 模拟超声深度衰减
    depth_factor = torch.linspace(1, 0.8, tensor.shape[-1])
    tensor *= depth_factor.view(1, 1, -1)
  
    return tensor.clamp(tensor.min(), tensor.max())

class ACDCDataset(Dataset):
    """ACDC数据集：加载ED/ES双时相+EF值+标签"""
    def __init__(self, case_paths, phase='train'):
        self.phase = phase
        self.case_info = [self.parse_case(p) for p in case_paths]

    def parse_case(self, case_dir):
        """解析病例信息：ED/ES帧、标签、EF值"""
        cfg_path = case_dir / 'Info.cfg'
        with open(cfg_path) as f:
            content = f.read()
      
        ed_frame = int(re.search(r'ED:\s*(\d+)', content).group(1))
        es_frame = int(re.search(r'ES:\s*(\d+)', content).group(1))
        label = CLASS_MAP[re.search(r'Group:\s*(\w+)', content).group(1)]

        # 加载ED/ES掩膜计算EF值
        ed_mask_path = case_dir / f"{case_dir.name}_frame{ed_frame:02d}_gt.nii.gz"
        es_mask_path = case_dir / f"{case_dir.name}_frame{es_frame:02d}_gt.nii.gz"
        ed_vol = self.calculate_volume(ed_mask_path)
        es_vol = self.calculate_volume(es_mask_path)
        ef = ((ed_vol - es_vol) / ed_vol * 100) if ed_vol !=0 else 0

        return {
            '4d_path': next(case_dir.glob('*_4d.nii.gz')),
            'ed_mask_path': ed_mask_path,
            'es_mask_path': es_mask_path,
            'label': label,
            'ed_frame': ed_frame - 1,
            'es_frame': es_frame - 1,
            'ef': ef
        }

    def calculate_volume(self, mask_path):
        """计算左心室体积（ml）"""
        mask_img = nib.load(mask_path)
        mask_data = mask_img.get_fdata()
        spacing = np.sqrt(np.sum(mask_img.affine[:3,:3]**2, axis=0))
        voxel_volume = np.prod(spacing)
        lv_voxels = (mask_data == 3).sum()
        return lv_voxels * voxel_volume / 1000

    def load_phase(self, info, phase):
        """加载并预处理单个时相（ED/ES）数据"""
        img_4d = nib.load(info['4d_path'])
        mask = nib.load(info[f'{phase}_mask_path']).get_fdata()
        phase_vol = img_4d.dataobj[..., info[f'{phase}_frame']]
        return self.preprocess(phase_vol, mask, img_4d.affine)

    def preprocess(self, volume, mask, affine):
        """3D医学图像预处理：重采样->归一化->ROI裁剪"""
        original_spacing = np.sqrt(np.sum(affine[:3,:3]**2, axis=0))
        zoom_factors = original_spacing / TARGET_SPACING
  
        # 重采样
        resampled_vol = zoom(volume, zoom_factors, order=3)
        resampled_mask = zoom(mask, zoom_factors, order=0)
  
        # 心脏ROI归一化
        heart_mask = (resampled_mask >= 1) & (resampled_mask <= 3)
        roi = resampled_vol[heart_mask]
        normalized = (resampled_vol - roi.mean()) / (roi.std() + 1e-8)
  
        # 智能裁剪
        return self.crop_heart(normalized, heart_mask)

    def crop_heart(self, volume, mask):
        """裁剪心脏ROI到目标尺寸"""
        bbox = find_objects(mask.astype(np.uint8))[0]
        if not bbox:
            return np.zeros(TARGET_SHAPE, dtype=volume.dtype)
      
        slices = [
            slice(max(0, bbox[i].start - 10), min(volume.shape[i], bbox[i].stop + 10))
            for i in range(3)
        ]
      
        source = volume[slices[0], slices[1], slices[2]]
        cropped = np.zeros(TARGET_SHAPE, dtype=volume.dtype)
      
        # 中心对齐填充
        y_start = (cropped.shape[0] - source.shape[0]) // 2
        x_start = (cropped.shape[1] - source.shape[1]) // 2
        z_start = (cropped.shape[2] - source.shape[2]) // 2
        y_end, x_end, z_end = y_start + source.shape[0], x_start + source.shape[1], z_start + source.shape[2]
      
        # 防止越界
        cropped[
            max(0, y_start):min(TARGET_SHAPE[0], y_end),
            max(0, x_start):min(TARGET_SHAPE[1], x_end),
            max(0, z_start):min(TARGET_SHAPE[2], z_end)
        ] = source[
            max(0, -y_start):min(source.shape[0], TARGET_SHAPE[0]-y_start),
            max(0, -x_start):min(source.shape[1], TARGET_SHAPE[1]-x_start),
            max(0, -z_start):min(source.shape[2], TARGET_SHAPE[2]-z_start)
        ]
        return cropped

    def __len__(self):
        if self.phase == 'train':
            return len(self.case_info) * (AUG_FACTOR + 1)
        return len(self.case_info)

    def __getitem__(self, idx):
        """返回：ED张量, ES张量, 标签, EF值"""
        # 增强策略
        if self.phase == 'train':
            transform = tio.Compose([
                PairwiseAugmentor(),
                tio.OneOf({
                    tio.RandomAffine(
                        scales=(0.9, 1.1), degrees=(-15, 15), translation=8,
                        isotropic=True, default_pad_value='minimum', image_interpolation='bspline'
                    ): 0.6,
                    tio.RandomElasticDeformation(num_control_points=5, max_displacement=5, locked_borders=1): 0.4,
                }),
                tio.RandomFlip(axes=(0, 1)),
                tio.Lambda(function=medical_intensity_augmentation),
                tio.RandomBlur(std=(0, 0.5)),
                tio.RandomNoise(std=(0, 0.08)),
                tio.RandomSwap(patch_size=15, num_iterations=3)
            ])
        else:
            transform = tio.Compose([])

        # 索引处理
        if self.phase == 'train':
            case_idx = idx // (AUG_FACTOR + 1)
            aug_idx = idx % (AUG_FACTOR + 1)
        else:
            case_idx = idx
            aug_idx = 0

        info = self.case_info[case_idx]
      
        # 加载ED/ES双时相
        ed_data = self.load_phase(info, 'ed').astype(np.float32)
        es_data = self.load_phase(info, 'es').astype(np.float32)
        ed_tensor = torch.from_numpy(ed_data).unsqueeze(0).float()
        es_tensor = torch.from_numpy(es_data).unsqueeze(0).float()
        ef_value = torch.tensor(info['ef'] / 100.0, dtype=torch.float32)
      
        # 同步增强（相同种子保证ED/ES变换一致）
        if self.phase == 'train' and aug_idx > 0:
            seed = case_idx * 1000 + aug_idx
            with torch.random.fork_rng():
                torch.manual_seed(seed)
                ed_tensor = transform(ed_tensor)
            with torch.random.fork_rng():
                torch.manual_seed(seed)
                es_tensor = transform(es_tensor)
      
        return ed_tensor, es_tensor, info['label'], ef_value

# ==================== 训练/评估函数（适配RCPE随机深度日志） ====================
def train_epoch(model, loader, criterion, optimizer):
   
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    keep_ratios = [] # 用于收集每个batch的RCPE保留率
  
    for ed_inputs, es_inputs, labels, ef in loader:
        # 双分支输入分别加载到设备
        ed_inputs = ed_inputs.to(DEVICE)
        es_inputs = es_inputs.to(DEVICE)
        labels = labels.to(DEVICE)
        ef = ef.float().to(DEVICE)
      
        optimizer.zero_grad()
        outputs = model(ed_inputs, es_inputs, ef)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)  # 梯度裁剪防止爆炸
        optimizer.step()
      
        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        #收集RCPE保留率
        keep_ratio = model.get_RCPE_keep_ratio()
        if keep_ratio is not None:
            keep_ratios.append(keep_ratio)
  
    avg_keep_ratio = float(np.mean(keep_ratios)) if len(keep_ratios) > 0 else 1.0
    return running_loss/len(loader), accuracy_score(all_labels, all_preds), avg_keep_ratio

def evaluate(model, loader, criterion):
    """评估函数（验证/测试）：仅保留损失与准确率"""
    model.eval()
    val_loss = 0.0
    val_preds, val_labels = [], []
  
    with torch.no_grad():
        for ed_inputs, es_inputs, labels, ef in loader:
            # 双分支输入分别加载到设备
            ed_inputs = ed_inputs.to(DEVICE)
            es_inputs = es_inputs.to(DEVICE)
            labels = labels.to(DEVICE)
            ef = ef.float().to(DEVICE)
          
            outputs = model(ed_inputs, es_inputs, ef)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * ed_inputs.size(0)
          
            _, preds = torch.max(outputs, 1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
  
    # 仅返回损失与准确率，与示例格式完全一致
    val_acc = accuracy_score(val_labels, val_preds)
    return val_loss/len(loader.dataset), val_acc

# ==================== 主训练流程（保留基于val_loss的逻辑） ====================
if __name__ == '__main__':
    # 多进程设置（Windows兼容）
    mp.set_start_method('spawn', force=True)
    
    # 配置参数
    start_fold = 0
    results_file = '-ED_ES_without_Diff.json' # 【可选】修改文件名区分实验
    CUSTOM_PREFIX = "-ED_ES_without_Diff"       # 【可选】修改前缀区分实验
    
    # 加载历史结果（避免重复训练）
    fold_results = []
    if os.path.exists(results_file):
        try:
            with open(results_file, 'r') as f:
                file_content = f.read().strip()
                if file_content:
                    fold_results = json.loads(file_content)
                    print(f"加载历史结果：{fold_results}")
        except json.JSONDecodeError:
            print("历史结果文件格式错误，重新开始")
    
    # 加载所有病例
    all_cases = [d for d in Path('ACDC/ACDC/database/training').glob('patient*') if d.is_dir()] + \
                [d for d in Path('ACDC/ACDC/database/testing').glob('patient*') if d.is_dir()]
    all_labels = []
    for case in all_cases:
        # 提取病例标签
        ed, es, label, ef = ACDCDataset([case], phase='val')[0]
        all_labels.append(label)
    
    # 五折分层交叉验证
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # 五折开始前计算一次模型复杂度（仅执行一次）
    model_temp = Caps3DNet(num_classes=5).to(DEVICE)
    calculate_model_complexity(model_temp, DEVICE)
    del model_temp  # 释放临时模型内存
    
    # 开始五折训练
    for fold, (train_val_idx, test_idx) in enumerate(kf.split(all_cases, all_labels)):
        # 跳过已训练的折数
        if fold < start_fold:
            continue
        print(f"\n=== 第 {fold+1}/5 折 ===")
      
        # 划分训练/验证/测试集
        train_val_cases = [all_cases[i] for i in train_val_idx]
        test_cases = [all_cases[i] for i in test_idx]
        train_val_labels = [all_labels[i] for i in train_val_idx]
      
        # 训练集内部再分层（75%训练，25%验证）
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
        for train_idx, val_idx in sss.split(train_val_cases, train_val_labels):
            train_cases = [train_val_cases[i] for i in train_idx]
            val_cases = [train_val_cases[i] for i in val_idx]
      
        # 创建数据集和加载器
        train_dataset = ACDCDataset(train_cases, phase='train')
        val_dataset = ACDCDataset(val_cases, phase='val')
        test_dataset = ACDCDataset(test_cases, phase='val')
      
        train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=0)
        test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=0)
      
        # 模型初始化
        model = Caps3DNet(num_classes=5, RCPE_drop_prob=0.05).to(DEVICE) # 显式传入drop_prob
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.AdamW(model.parameters(), lr=1e-3)
        
        # 学习率调度器统一看 val_loss
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
      
        # 训练跟踪变量
        best_loss = float('inf')  # 最佳模型跟踪指标改为 val_loss
        no_improve = 0
        best_model_path = f"{CUSTOM_PREFIX}_fold{fold}_best.pth"
      
        # 训练循环
        for epoch in range(100):
            # 接收三个返回值
            train_loss, train_acc, train_keep_ratio = train_epoch(model, train_loader, criterion, optimizer)
            # 评估函数仅返回损失和准确率
            val_loss, val_acc = evaluate(model, val_loader, criterion)
          
            # 打印格式增加RCPE keep ratio
            print(f"Epoch {epoch+1:2d} | RCPE keep ratio: {train_keep_ratio:.4f} | 训练损失: {train_loss:.4f} | 训练准确率: {train_acc:.2%} | 验证损失: {val_loss:.4f} | 验证准确率: {val_acc:.2%}")
          
            # 学习率调度基于 val_loss
            scheduler.step(val_loss)
          
            # 最佳模型保存基于 val_loss
            if val_loss < best_loss:
                best_loss = val_loss
                no_improve = 0
                torch.save(model.state_dict(), best_model_path)
                print(f"✅ 保存最佳模型 (验证损失降至 {val_loss:.4f})")
          
            # 早停（基于 val_loss）
            else:
                no_improve += 1
                if no_improve >= 10:
                    print(f"早停于Epoch {epoch+1}")
                    break
      
        # 测试集评估
        model.load_state_dict(torch.load(best_model_path))
        test_loss, test_acc = evaluate(model, test_loader, criterion)
        fold_results.append(test_acc)
        print(f"\n第 {fold+1} 折测试准确率: {test_acc:.2%}")
      
        # 保存结果（仅保存准确率）
        with open(results_file, 'w') as f:
            json.dump(fold_results, f)
        
        # 打印当前平均结果
        print(f"\n当前五折结果: {fold_results}")
        print(f"平均准确率: {np.mean(fold_results):.2%} (±{np.std(fold_results):.2%})")
    
    # 最终结果输出
    print("\n" + "="*60)
    print("=== 最终五折交叉验证结果（ED + ES without Diff消融） ===")
    print(f"五折结果: {fold_results}")
    print(f"平均准确率: {np.mean(fold_results):.2%} (±{np.std(fold_results):.2%})")
    print("="*60)